# 03. DAU (Daily Active Users)

GitHub Archive에서 "하루에 몇 명이 활동했는가?"를 구합니다.

DAU는 가장 기본적인 서비스 지표이자, 데이터 품질을 확인하는 첫 번째 수단입니다.

## 1. 데이터 로드

parquet 파일은 날짜별로 나뉘어 있으므로, 파일명에서 날짜를 붙여서 로드합니다.

In [ ]:
from pathlib import Path

import pandas as pd

from gharchive.transform import optimize_types

DATA_DIR = Path("../../data/daily_agg")

# 날짜별 parquet 로드 + date 컬럼 추가
frames = []
for path in sorted(DATA_DIR.glob("*.parquet")):
    df = pd.read_parquet(path)
    df["date"] = pd.to_datetime(path.stem, format="%Y%m%d").date()
    frames.append(df)

df = optimize_types(pd.concat(frames, ignore_index=True))
df["date"] = pd.to_datetime(df["date"])

print(f"Rows: {len(df):,}")
print(f"Date range: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"Days: {df['date'].nunique()}")

## 2. 전체 DAU

In [ ]:
from gharchive.stats import daily_active_users

dau = daily_active_users(df)

print(f"평균 DAU: {dau['dau'].mean():,.0f}")
print(f"최소 DAU: {dau['dau'].min():,} ({dau.loc[dau['dau'].idxmin(), 'date'].strftime('%a %m/%d')})")
print(f"최대 DAU: {dau['dau'].max():,} ({dau.loc[dau['dau'].idxmax(), 'date'].strftime('%a %m/%d')})")
print()
dau

## 3. DAU 추이 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(dau["date"], dau["dau"], marker="o", markersize=4, color="steelblue")
ax.set_title("GitHub DAU (Daily Active Users)")
ax.set_ylabel("Unique Users")
ax.set_xlabel("")

# 주말 표시
for _, row in dau.iterrows():
    if row["date"].weekday() >= 5:  # Saturday, Sunday
        ax.axvspan(row["date"] - pd.Timedelta(hours=12),
                   row["date"] + pd.Timedelta(hours=12),
                   alpha=0.08, color="gray")

ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 4. Event Type별 DAU

어떤 활동을 하는 유저가 많은가? Star만 누르는 유저 vs 코드를 Push하는 유저.

In [ ]:
from gharchive.stats import daily_active_users_by_type

dau_by_type = daily_active_users_by_type(df)

# 주요 이벤트만 시각화
TOP_TYPES = ["PushEvent", "WatchEvent", "CreateEvent", "IssueCommentEvent",
             "PullRequestEvent", "ForkEvent", "IssuesEvent"]

fig, ax = plt.subplots(figsize=(12, 5))
for etype in TOP_TYPES:
    sub = dau_by_type[dau_by_type["type"] == etype]
    ax.plot(sub["date"], sub["dau"], label=etype, linewidth=1.5)

ax.set_title("DAU by Event Type")
ax.set_ylabel("Unique Users")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 5. 요일별 DAU 패턴

In [ ]:
dau["weekday"] = dau["date"].dt.day_name()

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_avg = (
    dau.groupby("weekday")["dau"]
    .mean()
    .reindex(weekday_order)
)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["steelblue"] * 5 + ["coral"] * 2
weekday_avg.plot.bar(ax=ax, color=colors, edgecolor="black")
ax.set_title("Average DAU by Day of Week")
ax.set_ylabel("Avg Unique Users")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 6. 정리

- **DAU**: GitHub 전체 일별 활성 사용자 수
- **주말 효과**: 주말에 DAU가 눈에 띄게 감소 (개발자 = 직장인)
- **Event Type별 차이**: PushEvent 유저가 가장 많고, WatchEvent(Star)는 그보다 적음
- DAU 추이가 급등/급락하면 데이터 수집 오류나 특이 이벤트를 의심할 수 있음